In [10]:
import sempy.fabric as fabric
import time
import json
import base64

client = fabric.FabricRestClient()
workspace_id = fabric.resolve_workspace_id()

# Create Environment or get existing one
try:
    response = client.post(
        f"/v1/workspaces/{workspace_id}/environments",
        json={"displayName": "Env"}
    )
    env_id = response.json()["id"]
    print(f"Created Environment: {env_id}")

except Exception as e:
    if "400" in str(e) or "ItemDisplayNameAlreadyInUse" in str(e):
        print("Environment already exists, finding it...")
        envs = client.get(f"/v1/workspaces/{workspace_id}/environments").json()
        env_id = next(env["id"] for env in envs["value"] if env["displayName"] == "Env")
        print(f"Found existing Environment: {env_id}")
    else:
        raise

# Wait for any in-progress publish to finish
print("Checking publish status...")
while True:
    env_details = client.get(f"/v1/workspaces/{workspace_id}/environments/{env_id}").json()
    publish_state = env_details.get("properties", {}).get("publishDetails", {}).get("state", "")
    print(f"  Publish state: {publish_state}")
    if publish_state not in ["Running", "InProgress"]:
        break
    print("  Waiting for publish to complete...")
    time.sleep(15)

# Create environment.yml content with your libraries
yml_content = """dependencies:
  - pip:
    - semantic-link-labs
    - fabric-data-agent-sdk==0.1.9a0
    - tabulate==0.9.0
"""

# Upload the environment.yml file
lib_response = client.post(
    f"/v1/workspaces/{workspace_id}/environments/{env_id}/staging/libraries",
    files={"file": ("environment.yml", yml_content.encode(), "text/plain")}
)
print(f"Libraries uploaded: {lib_response.status_code} - {lib_response.text}")

# Publish the environment
publish_response = client.post(
    f"/v1/workspaces/{workspace_id}/environments/{env_id}/staging/publish"
)
print(f"Published: {publish_response.status_code}")

# Attach Env to all numbered notebooks via notebook definition update
print("\nAttaching Env to all notebooks...")
items = client.get(f"/v1/workspaces/{workspace_id}/items?type=Notebook").json()
for item in items["value"]:
    try:
        # Get notebook definition (async)
        def_resp = client.post(
            f"/v1/workspaces/{workspace_id}/items/{item['id']}/getDefinition"
        )

        # Poll until complete
        operation_id = def_resp.headers.get("x-ms-operation-id")
        while True:
            time.sleep(5)
            poll_resp = client.get(f"/v1/operations/{operation_id}")
            poll_json = poll_resp.json()
            state = poll_json.get("status", "")
            if state == "Succeeded":
                break
            elif state in ["Failed", "Cancelled"]:
                raise Exception(f"Operation failed: {poll_json}")

        # Get the result
        result_resp = client.get(f"/v1/operations/{operation_id}/result")
        def_json = result_resp.json()

        # Update environment in metadata
        parts = def_json.get("definition", {}).get("parts", [])
        for part in parts:
            if part["path"].endswith(".ipynb"):
                content = json.loads(base64.b64decode(part["payload"]).decode("utf-8"))
                content.setdefault("metadata", {}).setdefault("trident", {})["environment"] = {
                    "environmentId": env_id,
                    "workspaceId": workspace_id
                }
                part["payload"] = base64.b64encode(json.dumps(content).encode()).decode()

        # Update notebook definition
        update_resp = client.post(
            f"/v1/workspaces/{workspace_id}/items/{item['id']}/updateDefinition",
            json={"definition": def_json["definition"]}
        )
        status = "attached" if update_resp.status_code in [200, 202] else f"error {update_resp.status_code} - {update_resp.text}"
        print(f"  {status}: {item['displayName']}")

    except Exception as ex:
        print(f"  error on {item['displayName']}: {ex}")

StatementMeta(, 5bd4b39e-1cf6-4f06-9f39-1224570ee1a3, 12, Finished, Available, Finished, False)

Environment already exists, finding it...
Found existing Environment: 698b1279-733d-47d5-9f6f-bd9c05178597
Checking publish status...
  Publish state: Success
Libraries uploaded: 200 - {}
Published: 200

Attaching Env to all notebooks...
  attached: Load Notebooks
  attached: 2. Parameters
  attached: 3. M Code Highlighter
  attached: 4. Load Calc Dependency Data
  attached: 5. Data Transformation
  attached: 6. Load Tables Columns
  attached: 7. Load M Query
  attached: 8. Relationships
  attached: 9. Extract Column List
  attached: 10. Creating Semantic Model Diagram
  attached: 11. RLS
  attached: 12. Create Dim Database Table
  attached: 0. Create Lake House
  attached: 1. Run Notebooks
  attached: 0. Create Environment
